# Практика: современные CNN — домашняя работа


## Домашнее задание


### Архитектуры и статьи

Выберите одну архитектуру согласно своему варианту и изучите оригинальную статью.

| Архитектура | Ключевая идея | Оригинальная статья |
|---|---|---|
| **DenseNet-121 (2017)** | Плотные связи: каждый слой получает карты признаков всех предыдущих слоёв | [Densely Connected Convolutional Networks](https://openaccess.thecvf.com/content_cvpr_2017/html/Huang_Densely_Connected_Convolutional_CVPR_2017_paper.html) |
| **MobileNetV2 (2018)** | Inverted residual block, linear bottleneck и depthwise separable convolution | [MobileNetV2: Inverted Residuals and Linear Bottlenecks](https://openaccess.thecvf.com/content_cvpr_2018/html/Sandler_MobileNetV2_Inverted_Residuals_CVPR_2018_paper.html) |
| **ResNeXt-50 (2017)** | Group convolution, cardinality и схема split-transform-merge | [Aggregated Residual Transformations for Deep Neural Networks](https://openaccess.thecvf.com/content_cvpr_2017/html/Xie_Aggregated_Residual_Transformations_CVPR_2017_paper.html) |
| **GhostNet (2020)** | Получение дополнительных feature maps дешёвыми линейными преобразованиями | [GhostNet: More Features From Cheap Operations](https://openaccess.thecvf.com/content_CVPR_2020/html/Han_GhostNet_More_Features_From_Cheap_Operations_CVPR_2020_paper.html) |

### Задание

1. Выберите датасет для классификации изображений, связанный с транспортом. Опишите классы, количество объектов и способ разбиения данных.
2. Разделите данные на `train`, `validation` и `test`. Тестовую выборку используйте только для финальной оценки.
3. Реализуйте или соберите модель на основе архитектуры своего варианта и обучите её на выбранном датасете.
4. Посчитайте число обучаемых параметров модели.
5. Во время обучения сохраняйте `loss` и выбранную метрику отдельно для `train` и `validation`.
6. Постройте графики `train/validation loss` и `train/validation metric` по эпохам.
7. После выбора финальной модели один раз оцените её на `test` и приведите итоговые метрики.
8. Для многоклассовой задачи приведите как минимум `accuracy`, `macro F1` и confusion matrix. При дисбалансе классов дополнительно покажите `precision` и `recall` по классам.
9. Кратко опишите, какие идеи выбранной статьи реализованы в вашей модели и как они влияют на число параметров или вычислительную стоимость.

### Что сдавать

- ноутбук с воспроизводимым кодом;
- описание датасета и разбиения;
- описание архитектуры;
- графики обучения;
- итоговые метрики на `test`;
- краткие выводы и анализ ошибок модели.

[Форма для сдачи](https://forms.gle/e2a8o2ELmZ9693DUA)

## Мой вариант

ResNeXt-50 (32x4d). Датасет - GTSRB (дорожные знаки, 43 класса), та же версия с Kaggle, что и в
предыдущих домашках: она про транспорт и уже отлажена под эту задачу, искать новую смысла не было.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_random_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

## Датасет

GTSRB - немецкие дорожные знаки, 43 класса. Беру ту же Kaggle-версию, что и раньше: `Train/<class_id>/*.png`
и `Test/*.png` + `Test.csv`. Test в этом датасете выделен изначально, поэтому внутри train дополнительно
отделяю validation (80/20) - test трогаю только один раз, в самом конце, для финальной оценки.

In [ ]:
class GTSRBDataset(Dataset):
    """
    Читает GTSRB в раскладке с Kaggle. Все картинки декодируются и ресайзятся один раз
    в __init__ и хранятся в памяти как uint8-тензоры - иначе на каждой эпохе заново идет
    decode PNG с диска, и это очень долго.
    """

    def __init__(self, root, train=True, image_size=32):
        self.image_size = image_size
        self.images = []
        self.labels = []

        if train:
            train_dir = os.path.join(root, "Train")
            class_dirs = sorted(os.listdir(train_dir), key=lambda x: int(x))
            for class_dir in tqdm(class_dirs, desc="Загрузка train"):
                class_id = int(class_dir)
                class_path = os.path.join(train_dir, class_dir)
                for fname in os.listdir(class_path):
                    if not fname.endswith(".png"):
                        continue
                    img = Image.open(os.path.join(class_path, fname)).convert("RGB")
                    img = img.resize((image_size, image_size))
                    self.images.append(torch.from_numpy(np.array(img)).permute(2, 0, 1))
                    self.labels.append(class_id)
        else:
            test_csv = pd.read_csv(os.path.join(root, "Test.csv"))
            for _, row in tqdm(test_csv.iterrows(), total=len(test_csv), desc="Загрузка test"):
                img_path = os.path.join(root, row["Path"])
                img = Image.open(img_path).convert("RGB")
                img = img.resize((image_size, image_size))
                self.images.append(torch.from_numpy(np.array(img)).permute(2, 0, 1))
                self.labels.append(int(row["ClassId"]))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].float() / 255.0
        label = self.labels[idx]
        return image, label

In [ ]:
gtsrb_root = r"D:\coding\Studying\CV\Task2\GTSRB"

gtsrb_train_full = GTSRBDataset(gtsrb_root, train=True)
gtsrb_test = GTSRBDataset(gtsrb_root, train=False)

print(f"Train+val: {len(gtsrb_train_full)}, test: {len(gtsrb_test)}")

In [ ]:
train_labels = np.array(gtsrb_train_full.labels)
class_counts = np.bincount(train_labels, minlength=43)

plt.figure(figsize=(12, 3))
plt.bar(range(43), class_counts)
plt.xlabel("class_id")
plt.ylabel("количество картинок в train")
plt.title("Распределение классов GTSRB (train)")
plt.show()

print(f"Классов: {len(class_counts)}, min={class_counts.min()}, max={class_counts.max()}")

Классы заметно разбалансированы - у одних знаков в разы больше примеров, чем у других. Это важно держать в голове при финальной оценке: по accuracy такое легко не заметить.

In [ ]:
gtsrb_train_size = int(0.8 * len(gtsrb_train_full))
gtsrb_val_size = len(gtsrb_train_full) - gtsrb_train_size
gtsrb_train, gtsrb_val = random_split(gtsrb_train_full, [gtsrb_train_size, gtsrb_val_size])

gtsrb_train_loader = DataLoader(gtsrb_train, batch_size=64, shuffle=True)
gtsrb_val_loader = DataLoader(gtsrb_val, batch_size=64, shuffle=False)
gtsrb_test_loader = DataLoader(gtsrb_test, batch_size=64, shuffle=False)

## Модель - ResNeXt-50 (32x4d)

Идея ResNeXt - заменить обычную свёртку 3x3 внутри Bottleneck-блока на **групповую** свёртку
(`groups=cardinality`). Вместо того чтобы наращивать глубину или ширину сети, авторы статьи
наращивают cardinality - число независимых "путей" трансформации внутри одного блока (схема
split-transform-merge: вход делится на `cardinality` групп, каждая обрабатывается своей веткой,
результаты складываются). Групповая свёртка в PyTorch делает это одним вызовом
`nn.Conv2d(..., groups=cardinality)`, без ручного создания отдельных веток.

Ширину свёртки внутри блока считаю по формуле из статьи: `width = cardinality * base_width * (planes / 64)`.
Для варианта 32x4d это `cardinality=32`, `base_width=4` - веса внутри Bottleneck получаются заметно шире,
чем в обычном ResNet-50 (128 вместо 64 на первой стадии и так далее), а свёртка 3x3 всё равно остаётся
дешёвой за счёт группировки. Глубина сети (число блоков на каждой стадии) та же, что у ResNet-50 - `[3, 4, 6, 3]`.

In [ ]:
class ResNeXtBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, planes, stride=1, cardinality=32, base_width=4):
        super().__init__()
        width = int(planes * (base_width / 64.0)) * cardinality
        out_channels = planes * self.expansion

        self.conv1 = nn.Conv2d(in_channels, width, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=stride, padding=1,
                                groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out = out + identity
        return self.relu(out)

In [ ]:
class ResNeXtCIFAR(nn.Module):
    """
    Сборка ResNeXt в стиле CIFAR-версии ResNet из лекции: обычная свёртка 3x3 в начале
    вместо stem с stride 4, дальше четыре стадии из ResNeXtBottleneck-блоков.
    """

    def __init__(self, num_blocks, num_classes, cardinality=32, base_width=4):
        super().__init__()
        self.in_channels = 64
        self.cardinality = cardinality
        self.base_width = base_width

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * ResNeXtBottleneck.expansion, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResNeXtBottleneck(
                self.in_channels, planes, stride=s,
                cardinality=self.cardinality, base_width=self.base_width,
            ))
            self.in_channels = planes * ResNeXtBottleneck.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


def resnext50_cifar(num_classes, cardinality=32, base_width=4):
    return ResNeXtCIFAR([3, 4, 6, 3], num_classes, cardinality=cardinality, base_width=base_width)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
resnext_model = resnext50_cifar(num_classes=43).to(device)
print(f"Количество параметров: {count_parameters(resnext_model)}")

### Эффект группировки на число параметров

In [ ]:
# та же сборка, но cardinality=1 и base_width=64 - по формуле выше это дает обычный
# (не групповой) Bottleneck такой же ширины, как в классическом ResNet-50
plain_equivalent = resnext50_cifar(num_classes=43, cardinality=1, base_width=64)

resnext_params = count_parameters(resnext_model)
plain_params = count_parameters(plain_equivalent)

print(f"ResNeXt-50 (32x4d, групповая свертка): {resnext_params:,} параметров")
print(f"То же самое, но обычная свертка (cardinality=1): {plain_params:,} параметров")
print(f"Разница: {plain_params - resnext_params:,} параметров")

del plain_equivalent

Бутылочное горлышко у ResNeXt заметно шире (128/256/512/1024 вместо 64/128/256/512), но
групповая свертка 3x3 считается независимо внутри каждой из 32 групп, а не по всем каналам сразу - рост
ширины это почти полностью компенсирует. По факту у ResNeXt и у такого же блока без группировки итоговое
число параметров получается почти одинаковым (разница - пара процентов, см. число выше), хотя ширина
отличается вдвое. За счет группировки cardinality наращивается почти бесплатно по параметрам, вместо того
чтобы просто делать сеть глубже или шире - в этом и смысл трюка из статьи.

## Обучение

Сеть куда глубже всех предыдущих (50 слоёв), так что на CPU эпоха может идти ощутимо дольше - если
есть доступ к GPU (например в Colab), лучше гонять там.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, num_epochs, lr=1e-3, device="cpu"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Эпоха {epoch + 1}/{num_epochs}: "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    return history


def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Эпоха")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Эпоха")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
resnext_history = fit(resnext_model, gtsrb_train_loader, gtsrb_val_loader, num_epochs=10, lr=1e-3, device=device)

In [ ]:
plot_history(resnext_history)

## Финальная оценка на test

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy()


test_labels, test_preds = collect_predictions(resnext_model, gtsrb_test_loader, device)

test_accuracy = (test_labels == test_preds).mean()
test_f1_macro = f1_score(test_labels, test_preds, average="macro")

print(f"Accuracy на test: {test_accuracy:.4f}")
print(f"Macro F1 на test: {test_f1_macro:.4f}")

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(14, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
plt.title("Confusion matrix на test")
plt.show()

In [ ]:
print(classification_report(test_labels, test_preds, digits=3))

`classification_report` выше показывает precision/recall по каждому из 43 классов - раз количество
примеров по классам сильно разное (см. гистограмму в начале), общая accuracy может маскировать то, что
какие-то редкие знаки распознаются заметно хуже остальных. На confusion matrix стоит смотреть, какие
классы сеть путает чаще всего между собой - обычно это знаки, похожие по форме и цвету, различающиеся
только мелкими деталями внутри.

## Итог

Собрал ResNeXt-50 (32x4d) с нуля, в том же стиле, что и ResNet в лекции - Bottleneck с групповой сверткой
вместо обычной, сборка по стадиям `[3, 4, 6, 3]`. Посчитал число параметров и сравнил с таким же блоком без
группировки (разбор - в блоке выше): итоговая разница небольшая, при том что сама сборка заметно шире. Модель
обучена на GTSRB, train/val loss и accuracy считались по эпохам, графики построены. На test кроме accuracy
посчитаны macro F1, confusion matrix и precision/recall по классам - датасет разбалансирован, так что одной
accuracy тут не обойтись.